<div align="center">

**Taller Pandas**

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/e/ed/Pandas_logo.svg/240px-Pandas_logo.svg.png" alt="Pandas" width="240"/>

</div>

# Taller práctico: Pandas con Refinitiv LSEG

## Objetivo

Conectarte a **Refinitiv LSEG** desde Python y realizar consultas de datos utilizando **pandas**. Los resultados de las APIs de Refinitiv se obtienen en formato que se puede convertir fácilmente a DataFrame para exploración y análisis.

**Requisitos previos:**
- Eikon 4.0.36+ o Refinitiv Workspace 1.8+ instalado y abierto en tu equipo, **o** credenciales de Refinitiv Data Platform (RDP).
- App Key de Refinitiv (para sesión Desktop) o usuario/contraseña para RDP.

En este taller practicarás:
- Conectar a Refinitiv LSEG (sesión Desktop o Platform)
- Obtener datos (precios históricos, fundamentales) y convertirlos a DataFrame
- Explorar DataFrames (`.head()`, `.info()`, `.shape`, columnas)
- Trabajar con Series, filtrar y seleccionar datos
- Índices de fechas y rangos

---

## Conexión a Refinitiv LSEG

Antes de hacer consultas, hay que abrir una **sesión** con Refinitiv. Hay dos formas típicas:

1. **Sesión Desktop**: si tienes Eikon o Refinitiv Workspace abierto en el mismo equipo, usa tu **App Key**.
2. **Sesión Platform (RDP)**: si usas Refinitiv Data Platform en la nube, usa usuario, contraseña y `app_key`.

**Instalación:** `pip install refinitiv-data`

En la celda siguiente: reemplaza `"TU_APP_KEY"` por tu App Key real (o deja la variable para configurarla después). Si no tienes Refinitiv, puedes saltar esta celda y usar en los ejercicios las URLs de respaldo (S&P 500, FRED).

In [ ]:
# Conexión a Refinitiv LSEG (Desktop)
# Requiere: Eikon o Refinitiv Workspace abierto, y pip install refinitiv-data

import pandas as pd

USE_REFINITIV = False  # Cambiar a True cuando tengas App Key y Eikon/Workspace abierto
APP_KEY = "TU_APP_KEY"  # Reemplazar por tu App Key de Refinitiv

if USE_REFINITIV:
    try:
        import refinitiv.data as rd
        
        session = rd.session.desktop.Definition(app_key=APP_KEY).get_session()
        session.open()
        rd.session.set_default(session)
        print("Sesión Refinitiv abierta correctamente.")
    except Exception as e:
        print("No se pudo conectar a Refinitiv:", e)
        USE_REFINITIV = False
else:
    print("Sesión Refinitiv no activa (USE_REFINITIV=False). Usa URLs de respaldo en los ejercicios.")

### Obtener datos de Refinitiv y convertirlos a pandas DataFrame

Una vez conectado, puedes usar el **Content Layer** para precios históricos o datos fundamentales. El resultado tiene un atributo `.data.df` que es un **pandas DataFrame**.

**Ejemplo: precios históricos** (Historical Pricing Summaries). RIC de Ecopetrol en NYSE: `EC.N`; en Colombia: `ECOPETROL.CB`.

In [ ]:
# Ejemplo: obtener precios históricos de Ecopetrol (EC.N = NYSE) y cargar en pandas
# Solo se ejecuta si USE_REFINITIV es True y la sesión está abierta

if USE_REFINITIV:
    try:
        from refinitiv.data.content import historical_pricing
        
        response = historical_pricing.summaries.Definition(
            universe="EC.N",  # Ecopetrol ADR en NYSE
            interval=historical_pricing.Intervals.DAILY,
            count=90
        ).get_data()
        
        df_refinitiv = response.data.df  # pandas DataFrame
        print("DataFrame desde Refinitiv (primeras filas):")
        display(df_refinitiv.head(10))
    except Exception as e:
        print("Error al obtener datos de Refinitiv:", e)
        df_refinitiv = None
else:
    df_refinitiv = None
    print("Salta esta celda o activa USE_REFINITIV para obtener datos desde LSEG.")

## Ejercicio 1: Cargar y explorar una base de datos (Refinitiv o URL)

**Opción A (con Refinitiv):** Usa el DataFrame `df_refinitiv` obtenido en la celda anterior (precios históricos). Si no lo tienes, define `df = df_refinitiv.copy()` y trabaja con `df`.

**Opción B (sin Refinitiv):** Carga el listado de constituyentes del S&P 500 desde esta URL en un DataFrame `df`:
`https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv`

**Tareas (sobre el DataFrame `df` que elijas):**
1. Asegúrate de tener un DataFrame `df` (desde Refinitiv o desde la URL)
2. Muestra las dimensiones (`.shape`) y las primeras 10 filas (`.head(10)`)
3. Usa `.info()` para ver tipos de columnas
4. Lista los nombres de las columnas (`.columns`)

In [ ]:
# Opción A: usar DataFrame de Refinitiv (si ya ejecutaste la celda de precios históricos)
# Opción B: cargar S&P 500 desde URL
df_refinitiv = globals().get("df_refinitiv", None)
if df_refinitiv is not None and not df_refinitiv.empty:
    df = df_refinitiv.copy()
    print("Usando DataFrame desde Refinitiv LSEG.")
else:
    url_sp500 = "https://raw.githubusercontent.com/datasets/s-and-p-500-companies/master/data/constituents.csv"
    df = pd.read_csv(url_sp500)
    print("Usando DataFrame desde URL (S&P 500).")

# Exploración con pandas
print("Dimensiones:", df.shape)
print("\nPrimeras 10 filas:")
display(df.head(10))
print("\nTipos de columnas (.info()):")
df.info()
print("\nNombres de columnas:", df.columns.tolist())


## Ejercicio 2: Series y columnas

Usando el DataFrame `df` del Ejercicio 1:

**Si cargaste S&P 500 (constituyentes):**
1. Extrae la columna de sectores (p. ej. `GICS Sector`) como una Series
2. Usa `.value_counts()` para contar cuántas empresas hay por sector
3. Muestra los 5 sectores con más empresas
4. Obtén la columna de símbolos (tickers) y muestra los primeros 15 valores

**Si cargaste precios desde Refinitiv:** extrae la columna de cierre (p. ej. `Close` o `CLOSE`) como Series y calcula `.min()`, `.max()` y `.mean()`.

In [ ]:
# Según el origen del DataFrame (Refinitiv precios vs S&P 500), las columnas cambian
if "GICS Sector" in df.columns:
    serie_sector = df["GICS Sector"]
    print("Conteo por sector (.value_counts()):")
    print(serie_sector.value_counts().head(5))
    print("\nPrimeros 15 símbolos:")
    print(df["Symbol"].head(15))
else:
    # DataFrame de precios Refinitiv: buscar columna de cierre (puede ser Close, CLOSE, etc.)
    col_close = [c for c in df.columns if "close" in c.lower() or "CLOSE" in c]
    if col_close:
        serie = df[col_close[0]]
        print("Columna de cierre como Series - min, max, mean:")
        print(serie.min(), serie.max(), serie.mean())
    else:
        print("Columnas disponibles:", df.columns.tolist())


## Ejercicio 3: Filtrar datos

Sobre el mismo DataFrame `df`:

**Si es S&P 500 (constituyentes):**
1. Filtra solo las filas cuyo sector sea "Information Technology"
2. ¿Cuántas empresas de tecnología hay?
3. Filtra el sector "Financials" y muestra solo columnas Símbolo y Nombre (Symbol, Security)
4. Crea un sub-DataFrame con las primeras 20 filas y 3 columnas (Symbol, Security, GICS Sector)

**Si es precios Refinitiv:** filtra por rango de fechas con `.loc[]` (si el índice es fecha) o por una columna de fecha, y muestra las últimas 10 filas.

In [ ]:
# Filtrar según tipo de DataFrame
if "GICS Sector" in df.columns:
    tech = df[df["GICS Sector"] == "Information Technology"]
    print("Empresas de Information Technology:", len(tech))
    financials = df[df["GICS Sector"] == "Financials"][["Symbol", "Security"]]
    display(financials.head())
    sub = df.head(20)[["Symbol", "Security", "GICS Sector"]]
    display(sub)
else:
    # Precios Refinitiv: últimas 10 filas (o filtrar por fecha si hay columna Date)
    display(df.tail(10))


## Ejercicio 4: Serie temporal (índice de fechas)

**Opción A (Refinitiv):** Si ya tienes `df_refinitiv` de precios históricos, usa la columna de fecha como índice (`set_index`) y filtra con `.loc['2024-01-01':]`.

**Opción B (URL):** Carga el PIB trimestral de EE.UU. desde FRED:
`https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDP`

**Tareas:**
1. Lee el CSV con `parse_dates` en la primera columna (o usa el DataFrame de Refinitiv)
2. Asigna nombres claros a las columnas (p. ej. `fecha`, `pib`) si usas FRED
3. Usa `.set_index()` para poner la fecha como índice
4. Muestra `.head(10)` y `.tail(5)`
5. Filtra desde 2020 con `.loc['2020-01-01':]`

In [ ]:
# Opción A: usar precios Refinitiv con índice de fecha
# Opción B: cargar PIB desde FRED
df_ref = globals().get("df_refinitiv", None)
if df_ref is not None and not df_ref.empty:
    # Buscar columna de fecha en el DataFrame de Refinitiv
    date_cols = [c for c in df_ref.columns if "date" in c.lower() or "Date" in c]
    if date_cols:
        df_ts = df_ref.copy()
        df_ts[date_cols[0]] = pd.to_datetime(df_ts[date_cols[0]])
        df_ts = df_ts.set_index(date_cols[0])
    else:
        df_ts = df_ref.copy()
else:
    url_gdp = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=GDP"
    df_ts = pd.read_csv(url_gdp, parse_dates=[0])
    df_ts.columns = ["fecha", "pib_millones_usd"]
    df_ts = df_ts.set_index("fecha")

print("Head(10):")
display(df_ts.head(10))
print("Tail(5):")
display(df_ts.tail(5))
print("\nDesde 2020:")
display(df_ts.loc["2020-01-01":].head())


---

### Cerrar sesión Refinitiv (opcional)

Al terminar el taller, si abriste sesión con Refinitiv, puedes cerrarla con:

In [ ]:
# Solo si usaste sesión Refinitiv
if USE_REFINITIV:
    try:
        import refinitiv.data as rd
        rd.close_session()
        print("Sesión Refinitiv cerrada.")
    except Exception:
        pass

## Ejercicio 5: Crear Series y DataFrame desde cero

**Tareas:**
1. Crea una Series con los precios de cierre de 5 días: `[100, 102, 98, 105, 103]` e índice `['Lun','Mar','Mié','Jue','Vie']`
2. Crea un DataFrame con dos columnas: `['Fecha', 'Monto']` y al menos 4 filas de datos (por ejemplo, gastos o ingresos)
3. Muestra `.shape`, `.columns` e `.info()` del DataFrame creado

In [ ]:
# Escribe tu código aquí



## Ejercicio 6: Resumen

Elige una de las bases que cargaste (S&P 500 o PIB) y:

**Tareas:**
1. Obtén la columna numérica como Series y calcula `.min()`, `.max()`, `.mean()` (si aplica)
2. Para el DataFrame de S&P 500: ¿cuántas columnas son de tipo objeto/texto y cuántas numéricas? (usa `.dtypes`)
3. Muestra las últimas 10 filas del DataFrame de PIB con `.tail(10)`

In [ ]:
# Escribe tu código aquí

